In [ ]:
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm


In [ ]:
TRAINING_LOOP_DIR = Path(".").resolve()
REPO_DIR = TRAINING_LOOP_DIR.parent

DATASET_DIR = REPO_DIR.parent / "dataset"
TRAIN_DIR = DATASET_DIR / "Dataset_train"
VAL_DIR = DATASET_DIR / "Dataset_validation"

TRAIN_CSV = REPO_DIR / "tags" / "train.csv"
VAL_CSV = REPO_DIR / "tags" / "validation.csv"
OUTPUT_DIR = REPO_DIR / "output" / "history_2_5_d_simple_cnn"

label_columns = ["ICH"]

image_size = 224
num_windows = 128
window_size = 3

batch_size = 4
num_epochs = 50
learning_rate = 1e-4
weight_decay = 1e-4
num_workers = 4
prefetch_factor = 2
loss_log_every_i = 10
start_epoch = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train images: {TRAIN_DIR}")
print(f"Validation images: {VAL_DIR}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Validation CSV: {VAL_CSV}")
print(f"Output: {OUTPUT_DIR}")


In [ ]:
from dataset_2_5d import CTWindowsDataset


In [ ]:
train_dataset = CTWindowsDataset(
    table_path=TRAIN_CSV,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    image_size=image_size,
    num_windows=num_windows,
)

val_dataset = CTWindowsDataset(
    table_path=VAL_CSV,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    image_size=image_size,
    num_windows=num_windows,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

sample_windows, sample_label = train_dataset[0]
print(f"Windows shape: {sample_windows.shape}")
print(f"Label: {sample_label.tolist()}")


In [ ]:
class ConvBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm2d(out_channels)
        self.dropout = nn.Dropout2d(p=dropout_p) if dropout_p > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.InstanceNorm2d(out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        out = F.relu(self.norm1(self.conv1(x)), inplace=True)
        out = self.dropout(out)
        out = self.norm2(self.conv2(out))
        out = F.relu(out + identity, inplace=True)
        return out


class SliceEncoder2D(nn.Module):
    def __init__(self, in_channels=3, embedding_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock2D(in_channels, 16, stride=2, dropout_p=0.1),
            ResidualBlock2D(16, 16, stride=1, dropout_p=0.1),
            ResidualBlock2D(16, 32, stride=2, dropout_p=0.1),
            ResidualBlock2D(32, 32, stride=1, dropout_p=0.1),
            ResidualBlock2D(32, 64, stride=2, dropout_p=0.1),
            ResidualBlock2D(64, 64, stride=1, dropout_p=0.1),
            ResidualBlock2D(64, 128, stride=2, dropout_p=0.1),
            ResidualBlock2D(128, 128, stride=1, dropout_p=0.1),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, embedding_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)


class AttentionPooling(nn.Module):
    def __init__(self, embedding_dim=256, hidden_dim=128, dropout_p=0.3):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, embeddings):
        weights = torch.softmax(self.attention(embeddings), dim=1)
        pooled = torch.sum(weights * embeddings, dim=1)
        return pooled, weights.squeeze(-1)


class Simple2_5DClassifier(nn.Module):
    def __init__(self, in_channels=3, embedding_dim=256, num_classes=1):
        super().__init__()
        self.encoder = SliceEncoder2D(in_channels=in_channels, embedding_dim=embedding_dim)
        self.pooling = AttentionPooling(embedding_dim=embedding_dim, hidden_dim=128, dropout_p=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, windows):
        batch_size, num_windows, channels, height, width = windows.shape
        x = windows.view(batch_size * num_windows, channels, height, width)
        embeddings = self.encoder(x)
        embeddings = embeddings.view(batch_size, num_windows, -1)
        pooled, attention_weights = self.pooling(embeddings)
        logits = self.classifier(pooled)
        return logits, attention_weights


In [ ]:
torch.multiprocessing.set_sharing_strategy("file_system")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    prefetch_factor=prefetch_factor,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    prefetch_factor=prefetch_factor,
    pin_memory=(device.type == "cuda"),
)

model = Simple2_5DClassifier(in_channels=window_size, embedding_dim=256, num_classes=len(label_columns)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print(model)


In [ ]:
epoch_metrics_path = OUTPUT_DIR / "epoch_metrics.csv"
iter_losses_path = OUTPUT_DIR / "iter_losses_every_10.csv"

config = {
    "image_size": image_size,
    "num_windows": num_windows,
    "window_size": window_size,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "num_workers": num_workers,
    "prefetch_factor": prefetch_factor,
    "label_columns": label_columns,
    "train_csv": str(TRAIN_CSV),
    "val_csv": str(VAL_CSV),
    "train_dir": str(TRAIN_DIR),
    "val_dir": str(VAL_DIR),
}

with open(OUTPUT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)


for epoch in range(start_epoch, num_epochs + 1):
    epoch_started_at = time.time()
    model.train()
    epoch_loss_sum = 0.0
    iter_losses_current_epoch = []

    for i, (windows, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}", leave=False), start=1):
        windows = windows.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(windows)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        epoch_loss_sum += loss.item()

        if i % loss_log_every_i == 0:
            iter_losses_current_epoch.append(
                {
                    "epoch": epoch,
                    "split": "train",
                    "iteration": i,
                    "loss": round(loss.item(), 5),
                }
            )

    train_loss = round(epoch_loss_sum / max(len(train_loader), 1), 5)

    model.eval()
    val_loss_sum = 0.0

    with torch.no_grad():
        for windows, labels in val_loader:
            windows = windows.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits, _ = model(windows)
            loss = criterion(logits, labels)
            val_loss_sum += loss.item()

    val_loss = round(val_loss_sum / max(len(val_loader), 1), 5)
    epoch_time_sec = round(time.time() - epoch_started_at, 2)

    epoch_row = pd.DataFrame(
        [
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "epoch_time_sec": epoch_time_sec,
            }
        ]
    )

    epoch_row.to_csv(
        epoch_metrics_path,
        mode="a",
        header=not epoch_metrics_path.exists(),
        index=False,
    )

    if iter_losses_current_epoch:
        pd.DataFrame(iter_losses_current_epoch).to_csv(
            iter_losses_path,
            mode="a",
            header=not iter_losses_path.exists(),
            index=False,
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
        },
        OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt",
    )

